# Barcode Filtering Analysis

This notebook identifies barcodes that are present in all conditions for each strain and creates filtered CSV files containing only those common barcodes.

## Data Structure
- **ECI strain**: LCM, Pretreated (2 conditions)
- **ECJ strain**: LCM, Pretreated, GermFree (3 conditions)
- **ST69 strain**: LCM, Pretreated (2 conditions)
- **ST73 strain**: LCM, Pretreated, GermFree (3 conditions)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [2]:
%ls 

barcode_filter_analysis.ipynb  PCA_analysis_cleaned.ipynb
PCA_across_strains.ipynb


In [3]:
# Define file mappings for each strain and condition
strain_files = {
    'ECI': {
        'LCM': '../data/TV200_ECI_LCM.csv',
        'Pretreated': '../data/TV238_ECI_Pretreated.csv'
    },
    'ECJ': {
        'LCM': '../data/TV182_ECJ_LCM.csv',
        'Pretreated': '../data/TV240_ECJ_Pretreated.csv',
        'GermFree': '../data/TV248_ECJ_GermFree.csv'
    },
    'ST69': {
        'LCM': '../data/TV197_ST69_LCM.csv',
        'Pretreated': '../data/TV239_ST69_Pretreated.csv'
    },
    'ST73': {
        'LCM': '../data/TV199_ST73_LCM.csv',
        'Pretreated': '../data/TV241_ST73_Pretreated.csv',
        'GermFree': '../data/TV249_ST73_GermFree.csv'
    }
}

print("Strain file mappings:")
for strain, conditions in strain_files.items():
    print(f"{strain}: {list(conditions.keys())}")

Strain file mappings:
ECI: ['LCM', 'Pretreated']
ECJ: ['LCM', 'Pretreated', 'GermFree']
ST69: ['LCM', 'Pretreated']
ST73: ['LCM', 'Pretreated', 'GermFree']


In [8]:
def load_and_check_file(filepath):
    """Load a CSV file and return the barcode column as a set."""
    if not os.path.exists(filepath):
        print(f"Warning: File {filepath} not found")
        return set()
    
    try:
        # More efficient: only load the barcode column
        df = pd.read_csv(filepath, usecols=['barcode'])
        
        # Handle NaN values and get unique barcodes
        barcodes = df['barcode'].dropna().unique()
        print(f"{filepath}: {len(barcodes)} unique barcodes")
        return set(barcodes)
    except KeyError:
        print(f"Error: 'barcode' column not found in {filepath}")
        return set()
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return set()

def find_common_barcodes_for_strain(strain, conditions_dict):
    """Find barcodes present in all conditions for a given strain."""
    print(f"\n=== Processing {strain} strain ===")
    
    barcode_sets = []
    condition_names = []
    condition_counts = {}
    
    for condition, filename in conditions_dict.items():
        filepath = filename
        barcodes = load_and_check_file(filepath)
        if barcodes:
            barcode_sets.append(barcodes)
            condition_names.append(condition)
            condition_counts[condition] = len(barcodes)
    
    if not barcode_sets:
        print(f"No valid files found for {strain}")
        return set(), [], {}
    
    # Find intersection of all sets
    common_barcodes = barcode_sets[0]
    for barcode_set in barcode_sets[1:]:
        common_barcodes = common_barcodes.intersection(barcode_set)
    
    # Analysis: overlap statistics
    print(f"Barcode counts per condition:")
    for condition in condition_names:
        print(f"  {condition}: {condition_counts[condition]} barcodes")
    
    print(f"Common barcodes across all {strain} conditions: {len(common_barcodes)}")
    
    if len(condition_names) > 1:
        # Calculate retention rate
        min_barcodes = min(condition_counts.values())
        max_barcodes = max(condition_counts.values())
        retention_rate = len(common_barcodes) / max_barcodes * 100
        print(f"Retention rate: {retention_rate:.1f}% ({len(common_barcodes)}/{max_barcodes} from largest set)")
    
    return common_barcodes, condition_names, condition_counts

In [5]:
# Find common barcodes for each strain
strain_common_barcodes = {}

for strain, conditions in strain_files.items():
    common_barcodes, valid_conditions, condition_counts = find_common_barcodes_for_strain(strain, conditions)
    strain_common_barcodes[strain] = {
        'barcodes': common_barcodes,
        'conditions': valid_conditions,
        'files': conditions,
        'condition_counts': condition_counts
    }


=== Processing ECI strain ===
../data/TV200_ECI_LCM.csv: 39275 unique barcodes
../data/TV238_ECI_Pretreated.csv: 31011 unique barcodes
Barcode counts per condition:
  LCM: 39275 barcodes
  Pretreated: 31011 barcodes
Common barcodes across all ECI conditions: 27978
Retention rate: 71.2% (27978/39275 from largest set)

=== Processing ECJ strain ===
../data/TV182_ECJ_LCM.csv: 43697 unique barcodes
../data/TV240_ECJ_Pretreated.csv: 31563 unique barcodes
../data/TV248_ECJ_GermFree.csv: 29783 unique barcodes
Barcode counts per condition:
  LCM: 43697 barcodes
  Pretreated: 31563 barcodes
  GermFree: 29783 barcodes
Common barcodes across all ECJ conditions: 28612
Retention rate: 65.5% (28612/43697 from largest set)

=== Processing ST69 strain ===
../data/TV197_ST69_LCM.csv: 25724 unique barcodes
../data/TV239_ST69_Pretreated.csv: 25742 unique barcodes
Barcode counts per condition:
  LCM: 25724 barcodes
  Pretreated: 25742 barcodes
Common barcodes across all ST69 conditions: 25347
Retention r

In [6]:
# Display summary of common barcodes per strain
print("\n=== SUMMARY ===")
for strain, data in strain_common_barcodes.items():
    print(f"{strain}: {len(data['barcodes'])} common barcodes across {len(data['conditions'])} conditions")
    print(f"  Conditions: {', '.join(data['conditions'])}")


=== SUMMARY ===
ECI: 27978 common barcodes across 2 conditions
  Conditions: LCM, Pretreated
ECJ: 28612 common barcodes across 3 conditions
  Conditions: LCM, Pretreated, GermFree
ST69: 25347 common barcodes across 2 conditions
  Conditions: LCM, Pretreated
ST73: 23804 common barcodes across 3 conditions
  Conditions: LCM, Pretreated, GermFree


In [7]:
# Write out lists of common barcodes for each strain
for strain, data in strain_common_barcodes.items():
    if data['barcodes']:
        # Write list of common barcodes to a text file
        barcode_list_file = f"../data/{strain}_common_barcodes.txt"
        with open(barcode_list_file, 'w') as f:
            f.write(f"# Common barcodes for {strain} strain\n")
            f.write(f"# Found across conditions: {', '.join(data['conditions'])}\n")
            f.write(f"# Total count: {len(data['barcodes'])}\n\n")
            for barcode in sorted(data['barcodes']):
                f.write(f"{barcode}\n")
        
        print(f"Written {strain} common barcodes to: {barcode_list_file}")

Written ECI common barcodes to: ../data/ECI_common_barcodes.txt
Written ECJ common barcodes to: ../data/ECJ_common_barcodes.txt
Written ST69 common barcodes to: ../data/ST69_common_barcodes.txt
Written ST73 common barcodes to: ../data/ST73_common_barcodes.txt


In [9]:
def validate_common_barcodes(original_file, common_barcodes, strain, condition):
    """Validate that common barcodes have non-zero counts in the original data."""
    try:
        # Load only barcode and first few data columns to check for non-zero counts
        df = pd.read_csv(original_file)
        
        # Filter to common barcodes
        filtered_df = df[df['barcode'].isin(common_barcodes)]
        
        if filtered_df.empty:
            print(f"  WARNING: No common barcodes found in {strain} {condition}")
            return 0, 0
        
        # Check data columns (exclude barcode and new_locus_tag)
        data_columns = [col for col in filtered_df.columns if col not in ['barcode', 'new_locus_tag']]
        
        if not data_columns:
            print(f"  WARNING: No data columns found in {strain} {condition}")
            return len(filtered_df), 0
        
        # Count barcodes with non-zero counts in at least one sample
        non_zero_mask = (filtered_df[data_columns] > 0).any(axis=1)
        non_zero_count = non_zero_mask.sum()
        
        print(f"  {strain} {condition}: {non_zero_count}/{len(filtered_df)} common barcodes have non-zero counts")
        
        if non_zero_count < len(filtered_df):
            zero_only_count = len(filtered_df) - non_zero_count
            print(f"    WARNING: {zero_only_count} barcodes have only zero counts")
        
        return len(filtered_df), non_zero_count
        
    except Exception as e:
        print(f"  Error validating {strain} {condition}: {e}")
        return 0, 0

def create_filtered_csv(original_file, common_barcodes, output_file):
    """Create a new CSV file with only the common barcodes."""
    try:
        df = pd.read_csv(original_file)
        filtered_df = df[df['barcode'].isin(common_barcodes)]
        filtered_df.to_csv(output_file, index=False)
        print(f"Created {output_file} with {len(filtered_df)} barcodes (from {len(df)} original)")
        return True
    except Exception as e:
        print(f"Error creating {output_file}: {e}")
        return False

# Create filtered CSV files for each strain/condition combination
print("\n=== Creating filtered CSV files ===")

validation_summary = {}

for strain, data in strain_common_barcodes.items():
    if not data['barcodes']:
        print(f"Skipping {strain} - no common barcodes found")
        continue
    
    print(f"\nProcessing {strain} strain:")
    validation_summary[strain] = {}
    
    for condition, filename in data['files'].items():
        if condition in data['conditions']:  # Only process valid conditions
            # Validate common barcodes have non-zero counts
            total_common, non_zero_common = validate_common_barcodes(
                filename, data['barcodes'], strain, condition
            )
            validation_summary[strain][condition] = {
                'total_common': total_common,
                'non_zero_common': non_zero_common
            }
            
            # Create output filename
            base_name = filename.replace('.csv', '')
            output_file = f"{base_name}_filtered.csv"
            
            success = create_filtered_csv(filename, data['barcodes'], output_file)
            
            if not success:
                print(f"Failed to create filtered file for {strain} {condition}")


=== Creating filtered CSV files ===

Processing ECI strain:
  ECI LCM: 28058/28058 common barcodes have non-zero counts
Created ../data/TV200_ECI_LCM_filtered.csv with 28058 barcodes (from 39392 original)
  ECI Pretreated: 28058/28058 common barcodes have non-zero counts
Created ../data/TV238_ECI_Pretreated_filtered.csv with 28058 barcodes (from 31098 original)

Processing ECJ strain:
  ECJ LCM: 28675/28675 common barcodes have non-zero counts
Created ../data/TV182_ECJ_LCM_filtered.csv with 28675 barcodes (from 43785 original)
  ECJ Pretreated: 28675/28675 common barcodes have non-zero counts
Created ../data/TV240_ECJ_Pretreated_filtered.csv with 28675 barcodes (from 31632 original)
  ECJ GermFree: 28675/28675 common barcodes have non-zero counts
Created ../data/TV248_ECJ_GermFree_filtered.csv with 28675 barcodes (from 29850 original)

Processing ST69 strain:
  ST69 LCM: 25418/25419 common barcodes have non-zero counts
Created ../data/TV197_ST69_LCM_filtered.csv with 25419 barcodes (f

In [12]:
# Final summary with detailed analysis
print("\n=== FINAL SUMMARY ===")

print("=== Data Quality Analysis ===")
for strain, data in strain_common_barcodes.items():
    if not data['barcodes']:
        print(f"\n{strain}: NO COMMON BARCODES - All conditions have different barcode sets")
        continue
        
    print(f"\n{strain} Strain Analysis:")
    print(f"  Conditions analyzed: {', '.join(data['conditions'])}")
    print(f"  Common barcodes: {len(data['barcodes'])}")
    
    if data['condition_counts']:
        total_unique = sum(data['condition_counts'].values())
        avg_per_condition = total_unique / len(data['condition_counts'])
        print(f"  Average barcodes per condition: {avg_per_condition:.0f}")
        
        # Show filtering impact
        original_counts = list(data['condition_counts'].values())
        min_original = min(original_counts)
        max_original = max(original_counts)
        
        print(f"  Original barcode range: {min_original} - {max_original}")
        
        if strain in validation_summary:
            non_zero_counts = [v['non_zero_common'] for v in validation_summary[strain].values()]
            if non_zero_counts:
                avg_non_zero = sum(non_zero_counts) / len(non_zero_counts)
                print(f"  Avg common barcodes with data: {avg_non_zero:.0f}")

print("\n=== Files Created ===")
print("Common barcode lists:")
for strain in strain_common_barcodes.keys():
    if strain_common_barcodes[strain]['barcodes']:
        print(f"  ✓ {strain}_common_barcodes.txt")

print("\nFiltered CSV files:")
created_files = []
for strain, data in strain_common_barcodes.items():
    if data['barcodes']:
        for condition, filename in data['files'].items():
            if condition in data['conditions']:
                base_name = filename.replace('.csv', '')
                output_name = f"{base_name}_filtered.csv"
                created_files.append(output_name)
                print(f"  ✓ {output_name}")

print(f"\nTotal files created: {len([s for s in strain_common_barcodes.keys() if strain_common_barcodes[s]['barcodes']])} barcode lists + {len(created_files)} filtered CSV files")

# Summary statistics
print("\n=== Summary Statistics ===")
total_strains = len(strain_common_barcodes)
strains_with_common = len([s for s in strain_common_barcodes.values() if s['barcodes']])
print(f"Strains processed: {total_strains}")
print(f"Strains with common barcodes: {strains_with_common}")

if strains_with_common > 0:
    common_barcode_counts = [len(s['barcodes']) for s in strain_common_barcodes.values() if s['barcodes']]
    print(f"Common barcode counts: {min(common_barcode_counts)} - {max(common_barcode_counts)} (range)")
    print(f"Average common barcodes per strain: {sum(common_barcode_counts)/len(common_barcode_counts):.0f}")


=== FINAL SUMMARY ===
=== Data Quality Analysis ===

ECI Strain Analysis:
  Conditions analyzed: LCM, Pretreated
  Common barcodes: 27978
  Average barcodes per condition: 35143
  Original barcode range: 31011 - 39275
  Avg common barcodes with data: 28058

ECJ Strain Analysis:
  Conditions analyzed: LCM, Pretreated, GermFree
  Common barcodes: 28612
  Average barcodes per condition: 35014
  Original barcode range: 29783 - 43697
  Avg common barcodes with data: 28675

ST69 Strain Analysis:
  Conditions analyzed: LCM, Pretreated
  Common barcodes: 25347
  Average barcodes per condition: 25733
  Original barcode range: 25724 - 25742
  Avg common barcodes with data: 25418

ST73 Strain Analysis:
  Conditions analyzed: LCM, Pretreated, GermFree
  Common barcodes: 23804
  Average barcodes per condition: 24492
  Original barcode range: 23993 - 24880
  Avg common barcodes with data: 23870

=== Files Created ===
Common barcode lists:
  ✓ ECI_common_barcodes.txt
  ✓ ECJ_common_barcodes.txt
  ✓ 